### Gold Benchmarking - Baseline (Before Optimization)

### Purpose
This notebook captures baseline query performance on the Gold fact tables before applying:
- Liquid Clustering + OPTIMIZE
- Partitioning + ZORDER (comparative approach)

### Tables Used
- coffee.gold.fact_transactions
- coffee.gold.fact_transaction_items

### Instructions
For each query:
1. Run once (warm-up)
2. Run second time and note runtime
3. Capture query runtime and query profile (optional)

### Benchmark Queries
We use 5 business-driven benchmark queries:
1. Monthly Sales Trend
2. Store Performance (Last 3 Months)
3. Top 10 Customers by Spend
4. Top 10 Menu Items by Revenue
5. Join Drilldown (Store + Item Revenue)


In [0]:
-- =========================================================
-- Q1: Monthly Sales Trend
-- =========================================================
SELECT
  date_trunc('month', created_at) AS sales_month,
  ROUND(SUM(final_amount), 2) AS total_sales
FROM coffee.gold.fact_transactions
GROUP BY date_trunc('month', created_at)
ORDER BY sales_month;


In [0]:
-- =========================================================
-- Q2: Store Performance (Last 3 Months)
-- =========================================================
WITH max_dt AS (
  SELECT MAX(created_at) AS max_created_at
  FROM coffee.gold.fact_transactions
)
SELECT
  store_id,
  ROUND(SUM(final_amount), 2) AS total_sales
FROM coffee.gold.fact_transactions
WHERE created_at >= add_months((SELECT max_created_at FROM max_dt), -3)
GROUP BY store_id
ORDER BY total_sales DESC;

In [0]:
-- =========================================================
-- Q3: Top 10 Customers by Spend
-- =========================================================
SELECT
  item_id,
  ROUND(SUM(subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transaction_items
GROUP BY item_id
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- =========================================================
-- Q4: Top 10 Menu Items by Revenue
-- =========================================================
SELECT
  item_id,
  ROUND(SUM(subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transaction_items
GROUP BY item_id
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- =========================================================
-- Q5: Join Drilldown (Store + Item Revenue)
-- =========================================================
SELECT
  t.store_id,
  i.item_id,
  ROUND(SUM(i.subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transactions t
JOIN coffee.gold.fact_transaction_items i
  ON t.transaction_id = i.transaction_id
GROUP BY t.store_id, i.item_id
ORDER BY total_revenue DESC;

## Benchmark Results (Baseline)

| Query ID | Query Name | Runtime (Run 2) |
|---------|------------|-----------------|
| Q1 | Monthly Sales Trend | 2.23 |
| Q2 | Store Performance (Last 3 Months) |2.47  |
| Q3 | Top 10 Customers by Spend | 1.69 |
| Q4 | Top 10 Menu Items by Revenue | 1.59 |
| Q5 | Join Drilldown (Store + Item Revenue) | 6.19 |
